# Preprocess Forecast Data
This notebook fetches NOAA GEFS forecast data, computes quartiles, interpolates to hourly resolution, and saves to Zarr.

In [1]:
import xarray as xr
import pandas as pd
import numpy as np
from pathlib import Path
import logging
from tqdm import tqdm
import sys

# Add repo root to path to import utils
sys.path.append("../") 
from neuralhydrology.datautils.fetch_basin_forecasts import (
    load_basin_centroids,
    fetch_forecasts_for_basins,
    interpolate_to_hourly,
)

# Configuration
DATA_DIR = Path("../data/harz")
OUTPUT_ZARR = DATA_DIR / "timeseries" / "gefs_forecasts.zarr"
BASIN_CENTROIDS_FILE = DATA_DIR / "basin_centroids" / "basin_centroids.csv"

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()

In [2]:
# 1. Load Basin Centroids
if not BASIN_CENTROIDS_FILE.exists():
    raise FileNotFoundError(f"Basin centroids file not found: {BASIN_CENTROIDS_FILE}")

centroids = load_basin_centroids(BASIN_CENTROIDS_FILE)
print("Loaded centroids:", centroids)

Loaded centroids:   basin_name   latitude  longitude
0        DE1  51.834451  10.307828
1        DE2  51.816468  10.447276
2        DE3  51.810056  10.584058
3        DE4  51.752310  10.383225
4        DE5  51.884391  10.357203


In [3]:
# 2. Connect to NOAA GEFS
try:
    ds = xr.open_zarr(
        "https://data.dynamical.org/noaa/gefs/forecast-35-day/latest.zarr?email=optional@email.com", 
        decode_timedelta=True
    )
    print("Connected to NOAA GEFS Zarr.")
except Exception as e:
    print(f"Failed to connect to NOAA GEFS: {e}")

Connected to NOAA GEFS Zarr.


In [4]:
# 3. Process and Save per Basin
import time

# Define variables to fetch
VARIABLES = [
    "maximum_temperature_2m",
    "minimum_temperature_2m",
    "precipitation_surface",
    "relative_humidity_2m",
    "temperature_2m",
]
quartiles = [0.25, 0.5, 0.75]

print(f"Starting processing for {len(centroids)} basins...")

# Iterate over each basin
for i, (idx, row) in enumerate(tqdm(centroids.iterrows(), total=len(centroids))):
    basin_name = row['basin_name']
    
    # Retry logic for each basin
    max_retries = 10
    for attempt in range(1, max_retries + 1):
        try:
            # Select single basin centroid (keep as DataFrame to preserve structure)
            single_centroid = centroids.iloc[[i]]
            
            # Fetch
            # Note: We fetch only for this basin to minimize memory usage
            basin_forecast = fetch_forecasts_for_basins(ds[VARIABLES], single_centroid)
            
            # Compute Quartiles
            quartile_vars = {}
            for var in basin_forecast.data_vars:
                for q in quartiles:
                    q_str = f"_q{int(q*100)}"
                    # Drop the quantile coordinate to avoid merge conflicts
                    quartile_vars[f"{var}{q_str}"] = basin_forecast[var].quantile(q, dim='ensemble_member').drop_vars('quantile')
            
            ds_quartiles = xr.Dataset(quartile_vars, coords={k: v for k, v in basin_forecast.coords.items() if k != 'ensemble_member'})
            
            # Interpolate to hourly (first 10 days / 240 hours)
            ds_hourly = interpolate_to_hourly(ds_quartiles, max_hours=240)
            
            # Ensure basin dimension exists (fetch_forecasts might squeeze it if single basin)
            if 'basin' not in ds_hourly.dims:
                ds_hourly = ds_hourly.expand_dims('basin')
            
            # Save to Zarr
            if i == 0:
                # First basin: overwrite/create new store
                ds_hourly.to_zarr(OUTPUT_ZARR, mode='w')
            else:
                # Subsequent basins: append to the 'basin' dimension
                ds_hourly.to_zarr(OUTPUT_ZARR, mode='a', append_dim='basin')
                
            # Break retry loop on success
            break
            
        except Exception as e:
            print(f"Error processing basin {basin_name} (Attempt {attempt}/{max_retries}): {e}")
            if attempt < max_retries:
                time.sleep(5)
            else:
                print(f"Failed to process basin {basin_name} after {max_retries} retries.")
                raise e

print(f"Successfully saved all forecast data to {OUTPUT_ZARR}")

Starting processing for 5 basins...


 20%|██        | 1/5 [09:12<36:49, 552.34s/it]

Error processing basin DE2 (Attempt 1/10): 500, message='Internal Server Error', url='https://data.dynamical.org/noaa/gefs/forecast-35-day/latest.zarr?email=optional@email.com/precipitation_surface/c/25/0/0/0/2'
Error processing basin DE2 (Attempt 2/10): 500, message='Internal Server Error', url='https://data.dynamical.org/noaa/gefs/forecast-35-day/latest.zarr?email=optional@email.com/maximum_temperature_2m/c/1709/0/0/0/2'
Error processing basin DE2 (Attempt 2/10): 500, message='Internal Server Error', url='https://data.dynamical.org/noaa/gefs/forecast-35-day/latest.zarr?email=optional@email.com/maximum_temperature_2m/c/1709/0/0/0/2'
Error processing basin DE2 (Attempt 3/10): 500, message='Internal Server Error', url='https://data.dynamical.org/noaa/gefs/forecast-35-day/latest.zarr?email=optional@email.com/maximum_temperature_2m/c/122/0/0/0/2'
Error processing basin DE2 (Attempt 3/10): 500, message='Internal Server Error', url='https://data.dynamical.org/noaa/gefs/forecast-35-day/lates

 80%|████████  | 4/5 [49:41<11:33, 693.37s/it] 

Error processing basin DE5 (Attempt 1/10): 500, message='Internal Server Error', url='https://data.dynamical.org/noaa/gefs/forecast-35-day/latest.zarr?email=optional@email.com/relative_humidity_2m/c/1099/0/0/0/2'
Error processing basin DE5 (Attempt 2/10): 500, message='Internal Server Error', url='https://data.dynamical.org/noaa/gefs/forecast-35-day/latest.zarr?email=optional@email.com/maximum_temperature_2m/c/701/0/0/0/2'
Error processing basin DE5 (Attempt 2/10): 500, message='Internal Server Error', url='https://data.dynamical.org/noaa/gefs/forecast-35-day/latest.zarr?email=optional@email.com/maximum_temperature_2m/c/701/0/0/0/2'
Error processing basin DE5 (Attempt 3/10): 500, message='Internal Server Error', url='https://data.dynamical.org/noaa/gefs/forecast-35-day/latest.zarr?email=optional@email.com/maximum_temperature_2m/c/1218/0/0/0/2'
Error processing basin DE5 (Attempt 3/10): 500, message='Internal Server Error', url='https://data.dynamical.org/noaa/gefs/forecast-35-day/lates

100%|██████████| 5/5 [1:05:46<00:00, 789.32s/it]

Successfully saved all forecast data to ../data/harz/timeseries/gefs_forecasts.zarr
